In [1]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn


In [2]:
df = pd.read_csv(f"/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv", index_col = None)

In [3]:
df.shape

(7043, 21)

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
def clean_telco(df):
  df = df.drop("customerID", axis=1)
  df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
  df = df.dropna(subset = ["TotalCharges"])
  return df

df = clean_telco(df)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   object 
 16  PaymentMethod     7032 non-null   object 
 17  

In [7]:
def identify_column_types(df):
  column_types = {}
  for column in df.columns:
    if df[column].dtype == "object":
      if df[column].nunique() == df[column].count():
        column_types[column] = "drop"
      elif df[column].nunique() == 2:
        column_types[column] = "binary"
      elif df[column].nunique() >= 3:
        column_types[column] = "multi-category"

    elif df[column].dtype == "float64" or df[column].dtype == "int64":
      column_types[column] = "numeric"
  return column_types

col_types = identify_column_types(df)
print(col_types)



{'gender': 'binary', 'SeniorCitizen': 'numeric', 'Partner': 'binary', 'Dependents': 'binary', 'tenure': 'numeric', 'PhoneService': 'binary', 'MultipleLines': 'multi-category', 'InternetService': 'multi-category', 'OnlineSecurity': 'multi-category', 'OnlineBackup': 'multi-category', 'DeviceProtection': 'multi-category', 'TechSupport': 'multi-category', 'StreamingTV': 'multi-category', 'StreamingMovies': 'multi-category', 'Contract': 'multi-category', 'PaperlessBilling': 'binary', 'PaymentMethod': 'multi-category', 'MonthlyCharges': 'numeric', 'TotalCharges': 'numeric', 'Churn': 'binary'}


In [8]:
def encode_categoricals(df):
  for key, value in col_types.items():

    values = df[key].unique()

    if value == "binary":
      df[key] = df[key].map({values[0]: 1, values[1]: 0 })
    elif value == "multi-category":
      df = pd.get_dummies(df, columns=[key])
  return df
df = encode_categoricals(df)

In [9]:
df["Churn"] = df["Churn"].map({1: 0, 0: 1})

In [10]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,1,1,1,1,29.85,29.85,0,...,True,False,False,True,False,False,False,False,True,False
1,0,0,0,1,34,0,0,56.95,1889.50,0,...,True,False,False,False,True,False,False,False,False,True
2,0,0,0,1,2,0,1,53.85,108.15,1,...,True,False,False,True,False,False,False,False,False,True
3,0,0,0,1,45,1,0,42.30,1840.75,0,...,True,False,False,False,True,False,True,False,False,False
4,1,0,0,1,2,0,1,70.70,151.65,1,...,True,False,False,True,False,False,False,False,True,False


In [11]:
df.shape

(7032, 41)

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

In [24]:
X = df.drop("Churn", axis = 1)
y = df["Churn"]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [26]:
#random forest
rf = RandomForestClassifier(random_state = 42)
rf.fit(X_train, y_train)
rfpredict = rf.predict(X_test)
rfreport = classification_report(y_test, rfpredict)
print(rfreport)

              precision    recall  f1-score   support

           0       0.82      0.89      0.85      1033
           1       0.61      0.48      0.53       374

    accuracy                           0.78      1407
   macro avg       0.71      0.68      0.69      1407
weighted avg       0.77      0.78      0.77      1407



In [31]:
#logistic regression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr = scaler.transform(X_test)

LR = LogisticRegression(max_iter=1000, random_state = 42)
LR.fit(X_train_lr, y_train)
LRpredict = LR.predict(X_test_lr)
LRreport = classification_report(y_test, LRpredict)
print(LRreport)

              precision    recall  f1-score   support

           0       0.83      0.88      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407



In [32]:
#XGBoost
XGB = XGBClassifier(random_state = 42)
XGB.fit(X_train, y_train)
XGBpredict = XGB.predict(X_test)
XGBreport = classification_report(y_test, XGBpredict)
print(XGBreport)

              precision    recall  f1-score   support

           0       0.82      0.87      0.84      1033
           1       0.57      0.48      0.52       374

    accuracy                           0.77      1407
   macro avg       0.70      0.67      0.68      1407
weighted avg       0.76      0.77      0.76      1407



In [17]:
#DUE TO IMBALANCE RUN SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [22]:
#random forest SMOTE
rfsmote = RandomForestClassifier(random_state = 42)
rfsmote.fit(X_train_resampled, y_train_resampled)
rfpredictsmote = rfsmote.predict(X_test)
rfreportsmote = classification_report(y_test, rfpredictsmote)
print(rfreportsmote)

              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1033
           1       0.59      0.52      0.55       374

    accuracy                           0.78      1407
   macro avg       0.71      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407



In [36]:
#logistic regression smote
X_train_lr_smote = scaler.fit_transform(X_train_resampled)
X_test_lr_smote = scaler.transform(X_test)

LRsmote = LogisticRegression(max_iter=1000, random_state = 42)
LRsmote.fit(X_train_lr_smote, y_train_resampled)
LRpredictsmote = LRsmote.predict(X_test_lr_smote)
LRreportsmote = classification_report(y_test, LRpredictsmote)
print(LRreportsmote)

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      1033
           1       0.61      0.60      0.61       374

    accuracy                           0.79      1407
   macro avg       0.73      0.73      0.73      1407
weighted avg       0.79      0.79      0.79      1407



In [37]:
#XGBoost smote
XGBsmote = XGBClassifier(random_state = 42)
XGBsmote.fit(X_train_resampled, y_train_resampled)
XGBpredictsmote = XGBsmote.predict(X_test)
XGBreportsmote = classification_report(y_test, XGBpredictsmote)
print(XGBreportsmote)

              precision    recall  f1-score   support

           0       0.83      0.85      0.84      1033
           1       0.55      0.52      0.54       374

    accuracy                           0.76      1407
   macro avg       0.69      0.68      0.69      1407
weighted avg       0.76      0.76      0.76      1407

